# TP : Apprentissage Supervisé avec Python

## Atelier 4 : Feature Engineering et Classification

## Imports et configuration initiale

In [ ]:
import numpy as np
np.set_printoptions(threshold=10000, suppress=True)
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_curve, auc, make_scorer
from sklearn.pipeline import Pipeline
import pickle

## 1. Chargement des données et préparation

In [ ]:
df = pd.read_csv('Data/credit_scoring.csv', sep=';')

print("Premières lignes du dataset:")
print(df.head())
print("\nInformations sur le dataset:")
print(df.info())

In [ ]:
data = df.values
X = data[:, :-1]
Y = data[:, -1]

nom_cols = df.columns[:-1].values

print(f"Shape de X: {X.shape}")
print(f"Shape de Y: {Y.shape}")

In [ ]:
print(f"Taille de l'échantillon: {X.shape[0]} exemples")
print(f"Nombre de variables: {X.shape[1]}")

unique, counts = np.unique(Y, return_counts=True)
for val, count in zip(unique, counts):
    percentage = (count / len(Y)) * 100
    print(f"Classe {val}: {count} exemples ({percentage:.2f}%)")

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.5, random_state=1)

print(f"Taille du jeu d'entraînement: {X_train.shape[0]}")
print(f"Taille du jeu de test: {X_test.shape[0]}")

## 2. Fixer la variable SEED

In [ ]:
SEED = 1

## 3. Apprentissage et évaluation de modèles

In [ ]:
clfs = {
    'CART': DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'ID3': DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=SEED),
    'MLP': MLPClassifier(hidden_layer_sizes=(20, 10), random_state=SEED, max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Bagging': BaggingClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
}

In [ ]:
def evaluate_classifier(clf, X_train, Y_train, X_test, Y_test, clf_name="Classifier"):
    
    # Prédictions
    Y_pred = clf.predict(X_test)
    
    # Matrice de confusion
    cm = confusion_matrix(Y_test, Y_pred)
    print(f"\n{'='*60}")
    print(f"Résultats pour {clf_name}")
    print(f"{'='*60}")
    
    # Affichage textuel de la matrice de confusion
    print("\nMatrice de confusion:")
    print(cm)
    
    # Visualisation graphique de la matrice de confusion
    plt.figure(figsize=(8, 6))
    import seaborn as sns
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Classe 0', 'Classe 1'], 
                yticklabels=['Classe 0', 'Classe 1'])
    plt.ylabel('Vraie classe')
    plt.xlabel('Classe prédite')
    plt.title(f'Matrice de confusion - {clf_name}')
    plt.tight_layout()
    plt.show()
    
    # Métriques
    accuracy = accuracy_score(Y_test, Y_pred)
    precision = precision_score(Y_test, Y_pred, average='weighted', zero_division=0)
    recall = recall_score(Y_test, Y_pred, average='weighted', zero_division=0)
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    
    best_criterion = max(precision, recall)
    best_criterion_name = "Precision" if precision >= recall else "Recall"
    print(f"Meilleur critère: {best_criterion_name} = {best_criterion:.4f}")
    
    # Courbe ROC
    try:
        if hasattr(clf, 'predict_proba'):
            Y_proba = clf.predict_proba(X_test)
            if len(np.unique(Y_test)) == 2:
                fpr, tpr, _ = roc_curve(Y_test, Y_proba[:, 1])
                roc_auc = auc(fpr, tpr)
                
                plt.figure(figsize=(8, 6))
                plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
                plt.plot([0, 1], [0, 1], 'k--', label='Random')
                plt.xlabel('False Positive Rate')
                plt.ylabel('True Positive Rate')
                plt.title(f'Courbe ROC - {clf_name}')
                plt.legend()
                plt.grid(True)
                plt.show()
    except Exception as e:
        print(f"Impossible de tracer la courbe ROC: {e}")
    
    score_final = (accuracy + best_criterion) / 2
    print(f"\nScore final (accuracy + {best_criterion_name})/2: {score_final:.4f}")
    
    return score_final

In [ ]:
def run_classifiers_train_test(clfs, X_train, Y_train, X_test, Y_test):
    results = {}
    best_score = 0
    best_clf_name = None
    best_clf = None
    
    for clf_name, clf in clfs.items():
        print(f"\nEntraînement de {clf_name}...")
        clf.fit(X_train, Y_train)
        score = evaluate_classifier(clf, X_train, Y_train, X_test, Y_test, clf_name)
        results[clf_name] = score
        
        if score > best_score:
            best_score = score
            best_clf_name = clf_name
            best_clf = clf
    
    print("COMPARAISON DES SCORES FINAUX")
    for clf_name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
        print(f"{clf_name:15s}: {score:.4f}")
    
    print(f"\nMeilleur modèle: {best_clf_name} avec un score de {best_score:.4f}")
    
    return best_clf, best_clf_name, results

In [ ]:
print("ÉVALUATION SUR DONNÉES ORIGINALES")
best_clf_original, best_name_original, results_original = run_classifiers_train_test(
    clfs, X_train, Y_train, X_test, Y_test
)

## 4. Normalisation des variables continues

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Données normalisées!")
print(f"Moyenne des features (train): {X_train_scaled.mean(axis=0)[:5]}...")
print(f"Écart-type des features (train): {X_train_scaled.std(axis=0)[:5]}...")

In [ ]:
print("ÉVALUATION SUR DONNÉES NORMALISÉES")
best_clf_scaled, best_name_scaled, results_scaled = run_classifiers_train_test(
    clfs, X_train_scaled, Y_train, X_test_scaled, Y_test
)

In [ ]:
print(f"{'Classifier':<15} {'Original':<12} {'Normalisé':<12} {'Différence':<12}")
for clf_name in clfs.keys():
    diff = results_scaled[clf_name] - results_original[clf_name]
    print(f"{clf_name:<15} {results_original[clf_name]:<12.4f} {results_scaled[clf_name]:<12.4f} {diff:<12.4f}")

## 5. Création de nouvelles variables avec ACP

In [ ]:
pca = PCA(n_components=3, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Variance expliquée par chaque composante: {pca.explained_variance_ratio_}")
print(f"Variance totale expliquée: {pca.explained_variance_ratio_.sum():.4f}")

X_train_with_pca = np.concatenate([X_train_scaled, X_train_pca], axis=1)
X_test_with_pca = np.concatenate([X_test_scaled, X_test_pca], axis=1)

print(f"\nShape des données avec PCA: {X_train_with_pca.shape}")

In [ ]:
print("ÉVALUATION SUR DONNÉES NORMALISÉES + PCA")
best_clf_pca, best_name_pca, results_pca = run_classifiers_train_test(
    clfs, X_train_with_pca, Y_train, X_test_with_pca, Y_test
)

## 6. Validation croisée (Cross-Validation)

In [ ]:
def custom_score(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    return (acc + prec) / 2

custom_scorer = make_scorer(custom_score)

In [ ]:
def run_classifiers_cv(clfs, X, Y, cv_splits=10):
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=SEED)
    results = {}
    
    print(f"\nValidation croisée avec {cv_splits} folds")
    print("="*70)
    
    for clf_name, clf in clfs.items():
        cv_scores = cross_val_score(clf, X, Y, cv=kf, scoring=custom_scorer, n_jobs=-1)
        mean_score = np.mean(cv_scores)
        std_score = np.std(cv_scores)
        results[clf_name] = {'mean': mean_score, 'std': std_score, 'scores': cv_scores}
        print(f"{clf_name:<15} : {mean_score:.4f} +/- {std_score:.4f}")
    
    best_clf_name = max(results.items(), key=lambda x: x[1]['mean'])[0]
    print(f"\nMeilleur algorithme en CV: {best_clf_name}")
    
    return results

In [ ]:
print("VALIDATION CROISÉE - DONNÉES ORIGINALES")
cv_results_original = run_classifiers_cv(clfs, X, Y)

In [ ]:
print("VALIDATION CROISÉE - DONNÉES NORMALISÉES")
X_scaled_full = StandardScaler().fit_transform(X)
cv_results_scaled = run_classifiers_cv(clfs, X_scaled_full, Y)

In [ ]:
print("VALIDATION CROISÉE - DONNÉES NORMALISÉES + PCA")
pca_full = PCA(n_components=3, random_state=SEED)
X_pca_full = pca_full.fit_transform(X_scaled_full)
X_with_pca_full = np.concatenate([X_scaled_full, X_pca_full], axis=1)
cv_results_pca = run_classifiers_cv(clfs, X_with_pca_full, Y)

In [ ]:
print("SYNTHÈSE DES RÉSULTATS DE VALIDATION CROISÉE")
print(f"{'Classifier':<15} {'Original':<15} {'Normalisé':<15} {'Norm+PCA':<15}")
for clf_name in clfs.keys():
    orig = cv_results_original[clf_name]['mean']
    scaled = cv_results_scaled[clf_name]['mean']
    pca_result = cv_results_pca[clf_name]['mean']
    print(f"{clf_name:<15} {orig:<15.4f} {scaled:<15.4f} {pca_result:<15.4f}")

## 7. Paramétrage des classifieurs avec GridSearchCV

À adapter selon le meilleur algorithme identifié à l'étape 6

In [ ]:
def tune_hyperparameters(clf, param_grid, X_train, Y_train, cv_splits=5):
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=SEED)
    
    grid_search = GridSearchCV(
        clf, 
        param_grid, 
        cv=kf, 
        scoring=custom_scorer,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, Y_train)
    
    print("\nMeilleurs paramètres trouvés:")
    print(grid_search.best_params_)
    print(f"\nMeilleur score CV: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_, grid_search.best_params_

In [ ]:
print("OPTIMISATION DES HYPERPARAMÈTRES - RandomForest")

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

best_rf, best_params_rf = tune_hyperparameters(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid_rf,
    X_train_scaled,
    Y_train
)

In [ ]:
print("OPTIMISATION DES HYPERPARAMÈTRES - KNN")

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 10, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

best_knn, best_params_knn = tune_hyperparameters(
    KNeighborsClassifier(n_jobs=-1),
    param_grid_knn,
    X_train_scaled,
    Y_train
)

## 8. Sélection de variables

### Fonction 1: Importance des variables

In [ ]:
def plot_feature_importance(X_train, Y_train, feature_names):
    clf = RandomForestClassifier(n_estimators=1000, random_state=SEED, n_jobs=-1)
    clf.fit(X_train, Y_train)
    
    importances = clf.feature_importances_
    std = np.std([tree.feature_importances_ for tree in clf.estimators_], axis=0)
    sorted_idx = np.argsort(importances)[::-1]
    
    print("Importance des variables (ordre décroissant):")
    for i, idx in enumerate(sorted_idx[:10]):
        print(f"{i+1}. {feature_names[idx]}: {importances[idx]:.4f}")
    
    plt.figure(figsize=(10, 8))
    padding = np.arange(len(feature_names)) + 0.5
    plt.barh(padding, importances[sorted_idx], xerr=std[sorted_idx], align='center')
    plt.yticks(padding, feature_names[sorted_idx])
    plt.xlabel("Relative Importance")
    plt.title("Variable Importance")
    plt.tight_layout()
    plt.show()
    
    return importances, sorted_idx

In [ ]:
importances, sorted_idx = plot_feature_importance(X_train, Y_train, nom_cols)

### Fonction 2: Sélection du nombre optimal de variables

In [ ]:
def select_optimal_features(clf, X_train, Y_train, X_test, Y_test, sorted_idx):
    scores = np.zeros(X_train.shape[1])
    
    for f in range(X_train.shape[1]):
        X_train_f = X_train[:, sorted_idx[:f+1]]
        X_test_f = X_test[:, sorted_idx[:f+1]]
        
        clf.fit(X_train_f, Y_train)
        Y_pred = clf.predict(X_test_f)
        scores[f] = accuracy_score(Y_test, Y_pred)
    
    plt.figure(figsize=(12, 6))
    plt.plot(range(1, len(scores)+1), scores, marker='o')
    plt.xlabel("Nombre de Variables")
    plt.ylabel("Accuracy")
    plt.title("Evolution de l'accuracy en fonction des variables")
    plt.grid(True)
    plt.show()
    
    optimal_n_features = np.argmax(scores) + 1
    print(f"\nNombre optimal de variables: {optimal_n_features}")
    print(f"Meilleure accuracy: {scores[optimal_n_features-1]:.4f}")
    
    return scores, optimal_n_features

In [ ]:
best_algorithm = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
scores, optimal_n = select_optimal_features(
    best_algorithm, X_train, Y_train, X_test, Y_test, sorted_idx
)

### Fonction 3 (BONUS): Explicabilité avec SHAP

In [ ]:
import shap

def explain_with_shap(clf, X_train, X_test, feature_names):
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test)
    
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            shap_values_to_plot = shap_values[1]
            expected_value = explainer.expected_value[1]
        else:
            shap_values_to_plot = shap_values[1]
            expected_value = explainer.expected_value[1]
    else:
        shap_values_to_plot = shap_values
        expected_value = explainer.expected_value
    
    print("\nImportance globale des variables (SHAP):")
    shap.summary_plot(shap_values_to_plot, X_test, feature_names=feature_names, plot_type="bar")
    
    print("\nImpact des variables sur les prédictions:")
    shap.summary_plot(shap_values_to_plot, X_test, feature_names=feature_names)

print("EXPLICABILITÉ AVEC SHAP")

best_model_for_shap = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
best_model_for_shap.fit(X_train, Y_train)

explain_with_shap(best_model_for_shap, X_train, X_test, nom_cols)

## 9. Création d'un pipeline

In [ ]:
def create_pipeline(use_scaler=True, use_pca=False, n_components=3, clf=None, n_features=None):
    from sklearn.feature_selection import SelectKBest, f_classif
    
    steps = []
    
    if use_scaler:
        steps.append(('scaler', StandardScaler()))
    
    if n_features is not None:
        steps.append(('feature_selection', SelectKBest(f_classif, k=n_features)))
    
    if use_pca:
        steps.append(('pca', PCA(n_components=n_components, random_state=SEED)))
    
    if clf is None:
        clf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    steps.append(('classifier', clf))
    
    pipeline = Pipeline(steps)
    
    return pipeline

In [ ]:
print("CRÉATION DU PIPELINE FINAL")

final_pipeline = create_pipeline(
    use_scaler=True,
    use_pca=False,
    clf=RandomForestClassifier(**best_params_rf, random_state=SEED, n_jobs=-1),
    n_features=optimal_n
)

final_pipeline.fit(X_train, Y_train)

Y_pred_final = final_pipeline.predict(X_test)
final_accuracy = accuracy_score(Y_test, Y_pred_final)
final_precision = precision_score(Y_test, Y_pred_final, average='weighted', zero_division=0)
final_score = (final_accuracy + final_precision) / 2

print(f"\nPerformance du pipeline final:")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Score final: {final_score:.4f}")

In [ ]:
pipeline_filename = 'credit_scoring_pipeline.pkl'

with open(pipeline_filename, 'wb') as f:
    pickle.dump(final_pipeline, f)

print(f"\nPipeline sauvegardé dans: {pipeline_filename}")

with open(pipeline_filename, 'rb') as f:
    loaded_pipeline = pickle.load(f)

Y_pred_loaded = loaded_pipeline.predict(X_test)
assert np.array_equal(Y_pred_final, Y_pred_loaded), "Erreur: les prédictions ne correspondent pas!"
print("Pipeline chargé et vérifié avec succès!")

## Conclusion et résumé des résultats

In [ ]:
print("RÉSUMÉ DU TP - APPRENTISSAGE SUPERVISÉ")
print("\n1. CHARGEMENT DES DONNÉES")
print(f"   - Nombre d'exemples: {X.shape[0]}")
print(f"   - Nombre de variables: {X.shape[1]}")
print(f"   - Split train/test: 50/50")

print("\n2. ALGORITHMES TESTÉS")
for clf_name in clfs.keys():
    print(f"   - {clf_name}")

print("\n3. PRÉPARATION DES DONNÉES")
print("   - Données originales")
print("   - Données normalisées (StandardScaler)")
print("   - Données avec ACP (3 composantes)")

print("\n4. VALIDATION CROISÉE")
print("   - 10-fold cross-validation")
print("   - Score: (accuracy + precision) / 2")

print("\n5. OPTIMISATION")
print("   - GridSearchCV pour tuning des hyperparamètres")
print("   - Sélection de variables par importance")

print("\n6. PIPELINE FINAL")
print(f"   - Score final: {final_score:.4f}")
print(f"   - Sauvegardé dans: {pipeline_filename}")